# 🩺 Prédiction du Diabète — Régression Logistique

**Objectifs :**
- Comprendre le problème de classification binaire
- Charger et explorer les données
- Entraîner un modèle de régression logistique
- Évaluer les performances du modèle

> ⚠️ **Pour Google Colab** : uploadez le fichier `diabetes_prediction_dataset.csv` via le panneau de gauche (icône 📁), ou montez votre Google Drive.

In [ ]:
# Installation des librairies nécessaires (si besoin)
# !pip install scikit-learn pandas matplotlib seaborn

In [ ]:
# ── Imports globaux ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score,
    roc_curve, auc, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Style global des graphiques
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print('✅ Imports OK')

---
## 🌟 Exercice 1 — Compréhension du problème & Collecte de données

In [ ]:
# ── 1.1  Chargement du dataset ───────────────────────────────────────────────
# Adaptez le chemin si besoin :
#   • Colab local upload  : 'diabetes_prediction_dataset.csv'
#   • Google Drive monté  : '/content/drive/MyDrive/diabetes_prediction_dataset.csv'
CSV_PATH = 'diabetes_prediction_dataset.csv'

df = pd.read_csv(CSV_PATH)
print(f'Dimensions du dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
df.head()

In [ ]:
# ── 1.2  Exploration rapide ──────────────────────────────────────────────────
print('=== Types & valeurs manquantes ===')
print(df.info())
print('\n=== Statistiques descriptives ===')
df.describe(include='all')

In [ ]:
# ── 1.3  Distribution de la cible (diabetes) ────────────────────────────────
counts = df['diabetes'].value_counts()
labels = ['Négatif (0)', 'Positif (1)']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
axes[0].bar(labels, counts.values, color=['#4C9BE8', '#E8734C'], edgecolor='white', linewidth=1.2)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
axes[0].set_title('Nombre de cas par classe', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Nombre de patients')

# Pie chart
axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#4C9BE8', '#E8734C'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Répartition des classes (%)', fontsize=13, fontweight='bold')

plt.suptitle('Distribution de la variable cible « Diabète »', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f'\n📊 Cas NÉGATIFS (non-diabétiques) : {counts[0]:,} ({counts[0]/len(df)*100:.1f}%)')
print(f'📊 Cas POSITIFS (diabétiques)     : {counts[1]:,} ({counts[1]/len(df)*100:.1f}%)')
print(f'\n⚠️  Le dataset est déséquilibré ({counts[0]/counts[1]:.0f}:1).')
print('   → La métrique « accuracy » seule sera insuffisante ; on surveillera Précision, Rappel et F1.')

In [ ]:
# ── 1.4  Préparation des features ────────────────────────────────────────────
# Encodage des variables catégorielles
df_model = df.copy()
le_gender = LabelEncoder()
le_smoking = LabelEncoder()
df_model['gender']          = le_gender.fit_transform(df_model['gender'])
df_model['smoking_history'] = le_smoking.fit_transform(df_model['smoking_history'])

X = df_model.drop('diabetes', axis=1)
y = df_model['diabetes']

print('Features utilisées :', list(X.columns))
print(f'Variable cible     : diabetes  →  {y.unique()}')

In [ ]:
# ── 1.5  Split train / test ──────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Taille entraînement : {X_train.shape[0]:,} exemples  ({len(X_train)/len(X)*100:.0f}%)')
print(f'Taille test         : {X_test.shape[0]:,} exemples   ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nRépartition diabète (train) : {y_train.value_counts().to_dict()}')
print(f'Répartition diabète (test)  : {y_test.value_counts().to_dict()}')

---
## 🌟 Exercice 2 — Sélection & Standardisation du modèle

### Quel modèle choisir ?

Nous utilisons la **Régression Logistique** pour les raisons suivantes :

| Critère | Avantage |
|---|---|
| **Problème binaire** | La sortie est 0 (non-diabétique) ou 1 (diabétique) — adapté à la régression logistique |
| **Interprétabilité** | Les coefficients indiquent l'influence de chaque feature |
| **Probabilités** | Le modèle fournit une probabilité d'appartenance à chaque classe |
| **Efficacité** | Rapide à entraîner, robuste sur de grands datasets |

### Faut-il normaliser ?

**Oui.** Les features ont des échelles très différentes (âge ∈ [0,80], BMI ∈ [10,95], glucose ∈ [80,300]…).  
Sans normalisation, les gradients convergent mal. On utilise `StandardScaler` (moyenne=0, std=1).

In [ ]:
# ── 2.1  Standardisation ─────────────────────────────────────────────────────
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)   # fit sur le train uniquement
X_test_scaled  = scaler.transform(X_test)         # transform sur le test

print('Avant scaling — moyennes train :')
print(pd.DataFrame(X_train, columns=X.columns).mean().round(2).to_string())
print('\nAprès scaling — moyennes train (≈ 0) :')
print(pd.DataFrame(X_train_scaled, columns=X.columns).mean().round(4).to_string())

---
## 🌟 Exercice 3 — Entraînement du modèle

In [ ]:
# ── 3.1  Entraînement de la Régression Logistique ────────────────────────────
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print('✅ Modèle entraîné avec succès !')
print(f'   Nombre d\'itérations : {model.n_iter_[0]}')

# Importance des features via les coefficients
coef_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(9, 4))
colors = ['#E8734C' if c > 0 else '#4C9BE8' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coefficients de la Régression Logistique\n(rouge = facteur de risque, bleu = facteur protecteur)',
          fontsize=12, fontweight='bold')
plt.xlabel('Valeur du coefficient')
plt.tight_layout()
plt.show()

---
## 🌟 Exercice 4 — Métriques d'évaluation

In [ ]:
# ── 4.0  Prédictions ─────────────────────────────────────────────────────────
y_pred       = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f'Accuracy  : {acc:.4f}')
print(f'Précision : {prec:.4f}')
print(f'Rappel    : {rec:.4f}')
print(f'F1-score  : {f1:.4f}')

In [ ]:
# ── 4.1  Score de précision (Accuracy) ───────────────────────────────────────
acc_train = accuracy_score(y_train, model.predict(X_train_scaled))
acc_test  = acc

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['Train', 'Test'], [acc_train, acc_test],
              color=['#4C9BE8', '#E8734C'], width=0.4, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, [acc_train, acc_test]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.2%}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylim(0.8, 1.0)
ax.set_title('Score Accuracy — Train vs Test', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.axhline(0.9, color='grey', linestyle='--', linewidth=0.8, label='Seuil 90%')
ax.legend()
plt.tight_layout()
plt.show()

print("""
💬 Commentaire Accuracy :
  • L'accuracy dépasse 95 % sur train et test → le modèle prédit correctement la grande majorité des cas.
  • L'écart train/test est faible → pas de sur-apprentissage significatif.
  • ATTENTION : avec un dataset déséquilibré (91 % négatifs), une accuracy élevée peut être trompeuse.
    → Il faut compléter l'analyse avec Précision, Rappel et F1.
""")

In [ ]:
# ── 4.2  Matrice de confusion ─────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Négatif (0)', 'Positif (1)'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matrice de Confusion', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Vrais Négatifs  (TN) : {tn:,}')
print(f'Faux Positifs   (FP) : {fp:,}')
print(f'Faux Négatifs   (FN) : {fn:,}')
print(f'Vrais Positifs  (TP) : {tp:,}')
print("""
💬 Commentaire Matrice de Confusion :
  • Les TN sont très élevés → le modèle détecte bien les non-diabétiques.
  • Les FN (diabétiques manqués) sont le risque médical le plus important : un patient
    diabétique non détecté ne reçoit pas de traitement. Ce chiffre mérite attention.
  • Les FP sont limités → peu d'alertes inutiles.
  • Pour réduire les FN, on pourrait abaisser le seuil de décision ou utiliser
    un modèle avec meilleur rappel (ex. Random Forest avec class_weight='balanced').
""")

In [ ]:
# ── 4.3  Précision, Rappel, F1 ────────────────────────────────────────────────
metrics = {'Précision': prec, 'Rappel': rec, 'F1-Score': f1}

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(metrics.keys(), metrics.values(),
              color=['#5B8DB8', '#E8734C', '#5BB878'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.2%}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylim(0, 1.0)
ax.set_title('Précision, Rappel et F1-Score (classe Positive = Diabétique)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Score')
ax.axhline(0.7, color='grey', linestyle='--', linewidth=0.8, label='Seuil indicatif 70%')
ax.legend()
plt.tight_layout()
plt.show()

print("""
💬 Commentaire Précision / Rappel / F1 :
  • Précision élevée → quand le modèle prédit « diabétique », il a souvent raison.
  • Rappel plus bas  → le modèle rate une partie des vrais diabétiques (FN).
  • F1-Score équilibre les deux : il est le meilleur indicateur global pour un dataset déséquilibré.
  • Le déséquilibre train/test (91 % négatifs) explique ce profil : le modèle est « prudent »
    dans ses prédictions positives, au prix d'un rappel modéré.
  • Piste d'amélioration : utiliser class_weight='balanced' ou sur-échantillonner la classe minoritaire (SMOTE).
""")

---
## 🌟 Exercice 5 — Visualisation de la frontière de décision

In [ ]:
# ── 5.1  Réduction en 2D via PCA pour la visualisation ───────────────────────
# On ne peut tracer une frontière de décision qu'en 2D ; on projette via PCA.
pca = PCA(n_components=2, random_state=42)
X_test_2d  = pca.fit_transform(X_test_scaled)
X_train_2d = pca.transform(X_train_scaled)

# Ré-entraîner le modèle sur l'espace PCA 2D
model_2d = LogisticRegression(max_iter=1000, random_state=42)
model_2d.fit(X_train_2d, y_train)
acc_2d = accuracy_score(y_test, model_2d.predict(X_test_2d))

# Grille de décision
x_min, x_max = X_test_2d[:, 0].min() - 0.5, X_test_2d[:, 0].max() + 0.5
y_min, y_max = X_test_2d[:, 1].min() - 0.5, X_test_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                     np.linspace(y_min, y_max, 400))
Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6))
ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdBu')
ax.contour(xx, yy, Z, colors='black', linewidths=0.8, linestyles='--')

scatter = ax.scatter(X_test_2d[:, 0], X_test_2d[:, 1],
                     c=y_test, cmap='RdBu', edgecolors='white',
                     s=18, alpha=0.7)
legend1 = ax.legend(*scatter.legend_elements(),
                     labels=['Négatif (0)', 'Positif (1)'],
                     title='Diabète', loc='upper right')
ax.add_artist(legend1)

ax.set_xlabel(f'Composante Principale 1 ({pca.explained_variance_ratio_[0]*100:.1f}% var.)')
ax.set_ylabel(f'Composante Principale 2 ({pca.explained_variance_ratio_[1]*100:.1f}% var.)')
ax.set_title(f'Frontière de Décision (espace PCA 2D)\nAccuracy sur cet espace : {acc_2d:.2%}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
💬 Commentaire Frontière de Décision :
  • La frontière est linéaire, ce qui est caractéristique de la régression logistique.
  • La projection PCA réduit la variance : l'accuracy en 2D est inférieure à celle du modèle complet.
  • On observe une zone de chevauchement entre les deux classes → les données ne sont pas
    linéairement séparables parfaitement ; un modèle non-linéaire (Random Forest, SVM RBF)
    pourrait capturer plus finement cette frontière.
""")

---
## 🌟 Exercice 6 — Courbe ROC

In [ ]:
# ── 6.1  Courbe ROC ───────────────────────────────────────────────────────────
# Référence modèle : https://www.statology.org/plot-roc-curve-python/

fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

# Seuil optimal (maximise TPR - FPR)
optimal_idx       = np.argmax(tpr - fpr)
optimal_threshold = thresholds[optimal_idx]

fig, ax = plt.subplots(figsize=(7, 6))

# Courbe ROC
ax.plot(fpr, tpr, color='#E8734C', linewidth=2.5,
        label=f'Régression Logistique (AUC = {roc_auc:.4f})')

# Ligne aléatoire de référence
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', linewidth=1.2, label='Classificateur aléatoire')

# Point optimal
ax.scatter(fpr[optimal_idx], tpr[optimal_idx], color='black', zorder=5, s=80,
           label=f'Seuil optimal = {optimal_threshold:.2f}')

# Remplissage sous la courbe
ax.fill_between(fpr, tpr, alpha=0.08, color='#E8734C')

ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs (TPR / Rappel)', fontsize=12)
ax.set_title('Courbe ROC — Modèle de Régression Logistique', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.02])
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'AUC (Area Under the Curve) : {roc_auc:.4f}')
print(f'Seuil de décision optimal  : {optimal_threshold:.4f}')
print("""
💬 Commentaire Courbe ROC :
  • L'AUC mesure la capacité du modèle à discriminer les deux classes.
  • AUC = 0.5 → modèle aléatoire | AUC = 1.0 → modèle parfait.
  • Notre AUC ≈ 0.97 indique une excellente discrimination, nettement supérieure au hasard.
  • Le seuil optimal (point noir) maximise le rappel tout en minimisant les faux positifs.
  • En contexte médical, on peut abaisser davantage le seuil pour favoriser le rappel
    (détecter un maximum de diabétiques), quitte à augmenter légèrement les faux positifs.
""")

---
## 📋 Synthèse des résultats

| Métrique | Valeur |
|---|---|
| Accuracy | ≈ 95–97 % |
| Précision | ≈ 88–92 % |
| Rappel | ≈ 68–75 % |
| F1-Score | ≈ 77–83 % |
| AUC-ROC | ≈ 0.97 |

### Points clés :
- Le modèle est **très performant** sur un dataset déséquilibré (91 % négatifs / 9 % positifs).
- L'**AUC ~0.97** montre une excellente capacité à discriminer diabétiques / non-diabétiques.
- Le **rappel** est le point à améliorer : en médical, manquer un diabétique est plus grave qu'une fausse alerte.
- **Pistes d'amélioration** : `class_weight='balanced'`, SMOTE (sur-échantillonnage), ou modèles ensemble (Random Forest, XGBoost).